# RocketPy Basic Simulation Test

This notebook creates a basic rocketpy simulation to test out the most important aspect of the library. The code mainly follows the "first simulation" documentation of RocketPy: https://docs.rocketpy.org/en/latest/user/first_simulation.html

Note: to run the reference ARIS notebook, use the following two commands:
- docker build --no-cache -t asteria-ga:latest .
- docker run --gpus all -p 8888:8888 asteria-ga:latest conda run --no-capture-output -n adaenv jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root

In [ ]:
THRUST_CURVE_CSV = "../../rocketpy_simulation/data/ThrustCurve.csv"
DRAG_CURVE_CSV = "../../rocketpy_simulation/data/DragCurve.csv"
EUROC_PRESSURE_LEVELS_REANALYSIS_NC = "../../rocketpy_simulation/data/EuroC_pressure_levels_reanalysis_2021-2024.nc"

## Environment

In [ ]:
from rocketpy import Environment

env = Environment(latitude=39.389700, longitude=-8.288964, elevation=113, timezone="Europe/Portugal")
env.set_date((2023, 10, 14, 12))
env.set_atmospheric_model(type="Reanalysis", file=EUROC_PRESSURE_LEVELS_REANALYSIS_NC, dictionary="ECMWF")

env.max_expected_height = 10_000
env.info()

## Liquid Motor

In [ ]:
  print(env.density(160))    # should be ~1.225 kg/m³ at 160m
  print(env.pressure(160))   # should be ~101,000 Pa
  print(env.temperature(160)) # should be ~288 K

In [ ]:
from rocketpy import GenericMotor

# Motor parameters from config
dry_mass = 2.99  # kg
dry_inertia = (65.035, 64.931, 0.7440)
nozzle_radius = 0.051115125  # m
throat_radius = 0.034076749999999996  # m
nozzle_position = 0.0

# Tank masses (approximate from volumes & densities)
lox_mass = 1141 * 0.0126  # lox_density * oxidizer_tank_volume ≈ 13.692 kg
eth_mass = 789 * 0.009  # eth_density * fuel_tank_volume ≈ 7.101 kg
pressurant_mass = 303 * 0.0068  # pressurant_density_300bar * pressurant_tank_volume ≈ 2.0604 kg
propellant_mass = lox_mass + eth_mass + pressurant_mass

engine = GenericMotor(
    thrust_source=THRUST_CURVE_CSV,
    dry_mass=dry_mass,
    dry_inertia=dry_inertia,
    nozzle_radius=nozzle_radius,
    center_of_dry_mass_position=0,
    nozzle_position=nozzle_position,
    burn_time=None,  # inferred from thrust curve
    chamber_radius=0.085,  # tank_radius
    chamber_height=0.520 + 0.620 + 0.50051981,  # sum of tank heights (approx)
    chamber_position=1.07,  # approximate CoM of propellant system
    propellant_initial_mass=propellant_mass,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

## Rocket

In [ ]:
from rocketpy import Rocket

rocket = Rocket(
    radius=0.0895,
    mass=49,
    inertia=(94.51, 94.52, 0.2107),
    power_off_drag=DRAG_CURVE_CSV,
    power_on_drag=DRAG_CURVE_CSV,
    center_of_mass_without_motor=2.90,
    coordinate_system_orientation="nose_to_tail",
)

rocket.add_motor(engine, position=5.32)

rail_buttons = rocket.set_rail_buttons(
    upper_button_position=1.83116903,
    lower_button_position=5.0,
    angular_position=0.0,
)

nose_cone = rocket.add_nose(
    length=0.78100595,
    kind="Von Karman",
    position=0.0,
)

fin_set = rocket.add_trapezoidal_fins(
    n=4,
    root_chord=0.20272964694030443,
    tip_chord=0.11889239635534514,
    span=0.20848681179344536 - 0.0005500858676339606,
    position=4.708601669707605,
    cant_angle=0.5,
)

tail = rocket.add_tail(
    top_radius=0.0895,
    bottom_radius=0.08295,
    length=0.38,
    position=5.0,
)

# Drogue - deploys at apogee
drogue = rocket.add_parachute(
    name="Drogue",
    cd_s=0.7 * 1.0,  # drag coefficient  * area -> values taken from Slack channel
    trigger="apogee",
    sampling_rate=100,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

# Main - deploys at 2000m AGL
main = rocket.add_parachute(
    name="Main",
    cd_s=2.2 * 11.0,  # drag coefficient  * area -> values taken from Slack channel
    trigger=2000,
    sampling_rate=100,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

rocket.draw()

## Stability check

In [ ]:
rocket.plots.static_margin()

## Running the simulation

In [ ]:
from rocketpy import Flight

test_flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=15,
    inclination=85.0,
    heading=144.0,
)

In [ ]:
test_flight.prints.initial_conditions()

In [ ]:
test_flight.prints.surface_wind_conditions()

In [ ]:
test_flight.prints.launch_rail_conditions()


In [ ]:
test_flight.prints.out_of_rail_conditions()

In [ ]:
test_flight.prints.burn_out_conditions()


In [ ]:
test_flight.prints.apogee_conditions()


In [ ]:
test_flight.prints.events_registered()


In [ ]:
test_flight.prints.impact_conditions()


In [ ]:
test_flight.prints.maximum_values()


In [ ]:
test_flight.plots.trajectory_3d()

In [ ]:
test_flight.plots.linear_kinematics_data()


In [ ]:
test_flight.plots.flight_path_angle_data()


In [ ]:
test_flight.plots.attitude_data()


In [ ]:
test_flight.plots.angular_kinematics_data()


In [ ]:
test_flight.plots.aerodynamic_forces()


## Export flight data

In [ ]:
print(vars(test_flight))

In [ ]:
from rocketpy.simulation import FlightDataExporter

exporter = FlightDataExporter(test_flight)
exporter.export_data(
    "flight_data.csv",
    "x",
    "y",
    "z",
    "altitude",
    "acceleration",
)

In [ ]:
import plotly.express as px
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"

imported_data = pd.read_csv("flight_data.csv")
figure = px.line_3d(imported_data, x="X (m)", y="Y (m)", z="Z (m)")
figure.update_layout(width=700, height=700, margin=dict(l=0, r=0, b=0, t=0))
figure.show(renderer="browser")

print("Parachute Events: ")
for k, event in enumerate(test_flight.parachute_events):
    print(f"{k}: {event}")

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(imported_data["# Time (s)"], imported_data["Altitude AGL (m)"])

for k, event in enumerate(test_flight.parachute_events):
    print(f"{k}: {event}")
    ax.axvline(event[0], color="gray", alpha=0.5, linestyle="--")
    ax.text(event[0], ax.get_ylim()[1]-10, f"{k+1}. parachute", rotation=90, fontsize=10, va="top", ha="right", alpha=0.7)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(imported_data["# Time (s)"], imported_data["acceleration Magnitude (m/s²)"])

for k, event in enumerate(test_flight.parachute_events):
    print(f"{k}: {event}")
    ax.axvline(event[0], color="gray", alpha=0.5, linestyle="--")
    ax.text(event[0], ax.get_ylim()[1]-10, f"{k+1}. parachute", rotation=90, fontsize=10, va="top", ha="right", alpha=0.7)

ax.axvline(test_flight.rocket.motor.burn_out_time, color="red", alpha=0.3, linestyle="--")
ax.text(test_flight.rocket.motor.burn_out_time, ax.get_ylim()[1]-10, "Engine burn out", rotation=90, fontsize=10, va="top",
                ha="right", alpha=0.7)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(imported_data["# Time (s)"], imported_data["acceleration Magnitude (m/s²)"] / 9.81)
ax.set_xlabel("Time (s)")
ax.set_ylabel("G-Force (g)")
ax.set_title("G-Force over Time")

for k, event in enumerate(test_flight.parachute_events):
    ax.axvline(event[0], color="gray", alpha=0.5, linestyle="--")
    ax.text(event[0], ax.get_ylim()[1]-0.1, f"{k+1}. parachute", rotation=90, fontsize=10, va="top", ha="right", alpha=0.7)

ax.axvline(test_flight.rocket.motor.burn_out_time, color="red", alpha=0.3, linestyle="--")
ax.text(test_flight.rocket.motor.burn_out_time, ax.get_ylim()[1]-0.1, "Engine burn out", rotation=90, fontsize=10, va="top", ha="right", alpha=0.7)

ax.axhline(1, color="black", alpha=0.2, linestyle=":")
ax.text(imported_data["# Time (s)"].iloc[-1], 1.05, "1g", fontsize=9, alpha=0.5)
